# 1 Tone measurements -- 'CLI'

- All logic lives in the `qlab` package next to this notebook. Each cell is
**one measurement = one call**. Each call drops a timestamped CSV + PNG
into the dated data tree (in config) (see `README.md`).

- Everything here talks **straight to the N5222B over SCPI** —
`our code -> pyvisa -> VNA`. No PycQED and no QCoDeS anywhere: the
instrument is controlled and read directly, and the data comes straight back
to this PC.

- No signal generator connection yet (2 tone scans).

In [1]:
#set up --> change data output path in config
import os, sys
%matplotlib inline

#check in right folder
HERE = os.getcwd()
if not os.path.isdir(os.path.join(HERE, 'qlab')):
    LIB = r''      
    if LIB:        
        os.chdir(LIB); HERE = LIB
if HERE not in sys.path:
    sys.path.insert(0, HERE)

import qlab
st = qlab.connect_scpi()          

[qlab] direct-SCPI station: 'CH1_S11_1' on the N5222B at USB0::0x2A8D::0x2A01::MY58421887::0::INSTR.  Data -> C:/Data_HH/stub-s3


## Resonator spectroscopy 
- recreates source notebook scans through direct scpi communication --> no imported packages.
- Timing data is present for dev --> not necessary to run the scan

In [ ]:
# Low-power S21 scan  (QLab_tra cell 10)
import time

npts, avg = 2001, 200
t0 = time.perf_counter()
r = qlab.resonator_scan(st, center=8.01225e9, span=20e6, power=-30,
                        if_bandwidth=2000, npts=npts, averages=avg,
                        delay_t=6.5e-8, measure='S21')
dt = time.perf_counter() - t0
swp = st.vna.sweep_time()      # the instrument's own estimate for that sweep

print('saved:', r['csv_path'])
print(f'wall clock   : {dt:7.1f} s   ({dt/60:.2f} min)')
print(f'  of which   : {swp:7.1f} s   sweeping (the instrument, irreducible)')
print(f'  overhead   : {dt - swp:7.1f} s   transfer + CSV + plot (ours to shrink)')
print(f'per point    : {dt/npts*1e3:7.2f} ms')
print(f'per pt-avg   : {dt/(npts*avg)*1e6:7.1f} us  <- the rate to compare across settings')

In [ ]:
# S11 reflection  (QLab_tra cell 17)
import time

npts, avg = 2001, 100
t0 = time.perf_counter()
r = qlab.resonator_scan(st, center=8.22025e9, span=50e6, power=-30,
                        if_bandwidth=1000, npts=npts, averages=avg,
                        delay_t=qlab.config.DEFAULT_DELAY_S11, measure='S11')
dt = time.perf_counter() - t0
swp = st.vna.sweep_time()

print('saved:', r['csv_path'])
print(f'wall clock   : {dt:7.1f} s   ({dt/60:.2f} min)')
print(f'  of which   : {swp:7.1f} s   sweeping (the instrument, irreducible)')
print(f'  overhead   : {dt - swp:7.1f} s   transfer + CSV + plot (ours to shrink)')
print(f'per point    : {dt/npts*1e3:7.2f} ms')
print(f'per pt-avg   : {dt/(npts*avg)*1e6:7.1f} us  <- the rate to compare across settings')

In [3]:
# Punch-out: power sweep, dressed -> bare cavity  (QLab_tra cell 12)
#
# If must be stopped --> interrupt kernel over killing 
# If a read was in flight when you stopped it, run qlab.reset_link() before the next scan.
results = qlab.resonator_power_sweep(st, center=6.5e9, span=1e9,
                                     power_start=-30, power_stop=20, power_step=1,
                                     if_bandwidth=1000, npts=1001, averages=300,
                                     delay_t=6.5e-8, close_fig=True)
print(len(results), 'scans saved')

-30
  sweeping 1001 pts x 300 avg @ 1000 Hz IFBW -> ~276.9 s
resonator_scan_S21: min |S21| at 6.509000 GHz  ->  C:/Data_HH/stub-s3\20260722\122208_resonator_scan_S21\resonator_scan_S21.csv
-29
  sweeping 1001 pts x 300 avg @ 1000 Hz IFBW -> ~276.9 s
resonator_scan_S21: min |S21| at 6.325000 GHz  ->  C:/Data_HH/stub-s3\20260722\122646_resonator_scan_S21\resonator_scan_S21.csv
-28
  sweeping 1001 pts x 300 avg @ 1000 Hz IFBW -> ~276.9 s
resonator_scan_S21: min |S21| at 6.057000 GHz  ->  C:/Data_HH/stub-s3\20260722\123123_resonator_scan_S21\resonator_scan_S21.csv
-27
  sweeping 1001 pts x 300 avg @ 1000 Hz IFBW -> ~276.9 s
resonator_scan_S21: min |S21| at 6.016000 GHz  ->  C:/Data_HH/stub-s3\20260722\123601_resonator_scan_S21\resonator_scan_S21.csv
-26
  sweeping 1001 pts x 300 avg @ 1000 Hz IFBW -> ~276.9 s
resonator_scan_S21: min |S21| at 6.057000 GHz  ->  C:/Data_HH/stub-s3\20260722\124038_resonator_scan_S21\resonator_scan_S21.csv
-25
  sweeping 1001 pts x 300 avg @ 1000 Hz IFBW -> ~27

In [ ]:
# Long-term stability: rescan every 2 h  (QLab_tra cell 14)
# n_runs=None runs forever; set a number for a bounded test.
qlab.stability_monitor(st, center=8.25091e9, span=50e6, interval_s=7200, n_runs=3,
                       power=-30, if_bandwidth=1000, npts=2001, averages=100,
                       delay_t=6.24e-8, close_fig=True)

## 3. Shutdown

Two separate things, and the second is the one that used to be missing:

- `all_off(st)` — RF off, screen handed back to the front panel. Tidy-up.
- `disconnect(st)` — **closes the VISA session and releases the USB claim.**

USBTMC allows exactly one session per device, and that claim lives until the
owning *process* exits. Skipping `disconnect` is why a later `connect_scpi()`
hangs even after you shut every kernel down — Jupyter cannot kill a kernel
that is blocked inside a VISA read, so a zombie survives holding the device.

In [ ]:
qlab.all_off(st)      # VNA RF off, screen back to the front panel
qlab.disconnect(st)   # close the session, release the USB claim  <- do not skip

### Snapshot whatever the VNA is showing right now

**Read-only** 
Saves CSV + PNG into the dated tree like any other measurement.

The CSV carries a `screen_FDATA` column — the instrument's *own* formatted
values — next to our decode of SDATA. If those two agree, the data path is
confirmed end to end.

In [ ]:
snap = qlab.read_trace(st, name='vna_snapshot')

# Cross-check against a marker you've placed on the VNA screen:
import numpy as np
fq = 8.0122e9                      # <- the frequency your marker sits on
i  = np.argmin(np.abs(snap['freq_Hz'] - fq))
print(f"our value at {snap['freq_Hz'][i]/1e9:.6f} GHz = "
      f"{snap['amplitude_dB'][i]:.2f} dB   <-- compare to the marker readout")

### Stuck? Run `diagnose()` before changing anything

**A frozen screen is not the same as a broken link, and they need opposite fixes.**

*Frozen screen, stuck on an old span (e.g. 8.00225 → 8.02225 GHz).* This is
**cosmetic**. Taking a sweep requires `INIT:CONT OFF` so nothing free-running can
slip a stale sweep into your data, and that leaves the channel in HOLD — a held
PNA keeps redrawing the last trace it acquired. `resonator_scan()` now calls
`free_run()` the moment the data is read, so the screen goes live again after
every scan instead of staying stuck for the length of a punch-out. Nothing was
ever wrong with the link.

*The screen barely moves during a long scan.* Different thing, and not a fault.
With point averaging, one sweep dwells `averages` times on **every** point: 2001
points × 200 averages at 2 kHz IF bandwidth is minutes of real sweeping. Each
scan now prints the instrument's own sweep-time estimate before it triggers, so
you can tell "4 minutes to go" from "hung".

*Commands hang or return nonsense.* This is the real one. Two distinct causes:

| Cause | Tell | Fix |
|---|---|---|
| Device **claimed** by another session | open fails / hangs | see the 4 steps below |
| Link **dirty** — leftover bytes | connects, reads are garbage | `qlab.reset_link()` |

The dirty case: a `CALC:DATA?` read at 2001 points is one ~32 kB binary block. Cut
it short and the rest stays queued, so the next session's first query reads the
tail of the old trace as its answer. It accumulates per aborted run.

**Re-running the setup cell is now safe** — it closes the previous session first.
It used to leak one every time, and a single-session USB device with several
sessions on it behaves exactly like this. That is the most likely reason it
started only after you'd run a lot of things.

If opening still fails, no Python call can help — the claim is outside this process:

1. Task Manager → end every leftover `python.exe`
2. `services.msc` → restart **Keysight IO Libraries Service** ← survives shutting down all kernels
3. PNA front panel → **Preset**
4. System → Preferences → **Power On State: Preset** (not `Last State`, which restores the broken setup every reboot)

In [ ]:
# START HERE when something is wrong. Read-only, never raises, prints the whole
# picture: what VISA sees, how long the claim takes, what the channel is set to,
# and whether it is in HOLD (frozen screen) or sweeping. First FAIL is the one to chase.
qlab.diagnose()

In [ ]:
# Repair actions, once diagnose() has told you which one you need.
qlab.reset_link()          # dirty link: Device Clear + drain, then reconnect above

# qlab.free_run(st)              # frozen screen only (needs a working st)
# qlab.reset_link(preset=True)   # also SYST:PRES, if the channel state is wrong too

## 0. First-time hardware checks

In [ ]:
# Prove Python is controlling the VNA: changes the freq axis, asks you to look at the screen, then restores every setting it touched.  (REACH/TALK/OBEY/MEASURE)
#
# This opens its OWN session, so it closes `st` first (USBTMC allows one at a
# time). Re-run the setup cell above afterwards before measuring.
from check_control import run_control_check
run_control_check()               # pass no_pause=True to skip the screen prompts

In [ ]:
# Verify the fast binary transfer decodes correctly, using the VNA as its ownoracle. 
# eeds a trace defined on the VNA, set to log-mag. Want *_ok all True.
for k, v in st.vna.verify_against_instrument().items():
    print(f'{k:>18} : {v}')

### Check against the reference trace

`baseline_vna_snapshot.csv` next to this notebook is a known-good `read_trace`
run: 2001 points, 6–7 GHz, trace `CH1_S11_1`, taken 2026-07-21 11:50:54. In that
run the instrument's own formatted trace agreed with our decode, so the file is
evidence the data path was right at the time.

Two ways to use it, and they answer different questions:

| | needs the VNA? | answers |
|---|---|---|
| `compare_to_baseline()` | no | **does the code still do the same arithmetic?** |
| `compare_to_baseline(snap)` | yes | **is the setup still in the same state?** |

The first is the strict one — it must match to floating-point noise, so any
failure is a real code change. The second compares a fresh measurement against
an *uncalibrated* trace of standing waves in the cabling, so it only reproduces
if the VNA is in the same front-panel state and nothing was recabled. A
mismatch there is information, not necessarily a bug.

In [ ]:
# Regression check on the decode math. No VNA needed — runs anywhere.
# Every line must say OK; if one doesn't, the code changed, not the hardware.
qlab.compare_to_baseline()

In [ ]:
# Live check: the snapshot above vs the reference measurement.
# Needs the VNA set to the same 6-7 GHz / 2001-point CH1_S11_1 trace.
# Loosen tol_dB if the trace is noisier than the reference was.
qlab.compare_to_baseline(snap, tol_dB=0.5, tol_Hz=5e6)

In [ ]:
# DIAGNOSTIC: is there a real path, or am I looking at the receiver noise floor?
# Run this when a scan comes back flat/featureless. ~30 s, no library changes.
#
# |S21| is a RATIO, and that is what makes this work:
#   real transmission path -> the signal scales with the drive, so the ratio HOLDS
#   receiver noise floor   -> the numerator is fixed noise, so the ratio FALLS by
#                             exactly the power increase
# No cabling fault can fake that, which is why this separates "nothing connected"
# from "connected but cold/unpowered/out of band" when the trace alone cannot.
#
# Everything is held fixed except power. Small and fast on purpose — we want a
# level, not resolution. Keep `averages` identical between the two runs: vector
# averaging of pure noise pulls the mean toward zero, so the apparent floor
# depends on the averaging, and only a like-for-like comparison is meaningful.
# 0 dBm is well inside what the punch-out cell below already sweeps (-30 -> +20).
import numpy as np

probe = dict(center=8.01225e9, span=20e6, npts=201, averages=10,
             if_bandwidth=2000, delay_t=6.5e-8, measure='S21')

levels = {}
for p in (-30, 0):
    r = qlab.resonator_scan(st, power=p, analyze=False, **probe)
    # median, not mean: at the noise floor the dB values spike to -120 wherever
    # the complex average lands near zero, and a mean chases those outliers.
    levels[p] = float(np.median(r['amplitude_dB']))
    print(f'  {p:+4d} dBm  ->  median |S21| = {levels[p]:7.1f} dB')

drop = levels[-30] - levels[0]
print(f'\ndrive raised 30 dB, |S21| ratio fell {drop:.1f} dB')
if drop > 20:
    print('=> NOISE FLOOR. No port1 -> port2 path at this frequency. Check the')
    print('   cabling, whether the HEMT is powered, and whether the fridge is cold.')
elif drop < 10:
    print('=> REAL PATH, just lossy. The ratio held, so signal scaled with drive.')
    print('   The resonance is absent for another reason — warm device, or the')
    print('   feature is outside this 20 MHz window. Widen the span next.')
else:
    print('=> AMBIGUOUS. Possibly compressing, or marginally above the floor.')
    print('   Repeat with if_bandwidth=200: that drops the floor ~10 dB and')
    print('   leaves a real ratio untouched — same logic, independent knob.')

## 2. Not here yet — the pump-based measurements

Two-tone, the pump sweeps, and the Kerr test all drive the **Anritsu MG3692C**
pump, which used to go through PycQED/QCoDeS. That path was **removed** rather
than left sitting in the code as a hidden dependency — this library is now
100% direct SCPI.

They come back once the Anritsu is connected and we have its programming
manual, as a direct SCPI driver alongside `scpi_vna.py`. Nothing is lost: the
original code is in git history and in the two frozen notebooks. See the
README roadmap.